In [ ]:
import time

import collections.abc 
collections.Iterable=collections.abc.Iterable

## Robot packages:
import urx  #The UR remote control library.
import math3d as m3d 
import sys
import socket

In [ ]:
HOST = "192.168.1.10"
PORT = 30002
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect((HOST, PORT))

#This is the function that is used to move the gripper width
def gripper_width(width):
    t_sleep = 0.05
    #First set all DIO to false
    time.sleep(t_sleep)
    s.send(("set_digital_out(0,False)" + "\n").encode())
    time.sleep(t_sleep)
    s.send(("set_digital_out(1,False)" + "\n").encode())
    time.sleep(t_sleep)
    s.send(("set_digital_out(2,False)" + "\n").encode())
    time.sleep(t_sleep)
    s.send(("set_digital_out(3,False)" + "\n").encode())
    time.sleep(t_sleep)
    s.send(("set_digital_out(4,False)" + "\n").encode())   #Added this so the gripper only starts moving when dio 4 = false
    time.sleep(t_sleep)
    s.send(("set_digital_out(5,False)" + "\n").encode())   #Added this so the gripper only starts moving when dio 4 = false
    time.sleep(t_sleep)
    s.send(("set_digital_out(6,False)" + "\n").encode())   #Added this so the gripper only starts moving when dio 4 = false
    time.sleep(t_sleep)
    s.send(("set_digital_out(7,False)" + "\n").encode())   #Added this so the gripper only starts moving when dio 4 = false
    time.sleep(t_sleep)
    
    if width == 0:
        s.send(("set_digital_out(0,True)" + "\n").encode())
    elif width == 20:
        s.send(("set_digital_out(1,True)" + "\n").encode())
    elif width == 50:
        s.send(("set_digital_out(2,True)" + "\n").encode())
    elif width == 70:
        s.send(("set_digital_out(3,True)" + "\n").encode())
    elif width == 100:
        s.send(("set_digital_out(4,True)" + "\n").encode())
    elif width == 60:
        s.send(("set_digital_out(5,True)" + "\n").encode())
    elif width == 65:
        s.send(("set_digital_out(6,True)" + "\n").encode()) 
    elif width == 95:
        s.send(("set_digital_out(7,True)" + "\n").encode())   
    else:
        print("Width not defined")

In [ ]:
robot = urx.Robot("192.168.1.10", use_rt=True)

In [ ]:
robot.getj()

In [ ]:
gripper_width(100)

In [ ]:
dir = '/home/avi/Desktop/robomason/_workingdata/_constructionruns/construction_run_05/raw_data.pkl'

import zmq
import pickle
import time
import msgpack

# Initialize ZeroMQ context and socket
context = zmq.Context()
zmq_socket = context.socket(zmq.PUB)
zmq_socket.bind("tcp://127.0.0.1:5555")

with open(dir, 'rb') as file:
    data_list = pickle.load(file)

# Parameters
playback_speed = 7.5  # Adjust the playback speed
send_interval = 10# Send only every 'n' packets (change this as needed)

# Initialize variables
previous_timestamp = None
packet_counter = 0  # Counter to track packets

# Loop through the data
for data in data_list:
    # Calculate delay to simulate real-time data
    if previous_timestamp is not None:
        delay = (data['timestamp_send'] - previous_timestamp) / playback_speed
        if delay > 0:
            time.sleep(delay)
    previous_timestamp = data['timestamp_send']

    # Increment packet counter
    packet_counter += 1

    # Send only every 'n' packets
    if packet_counter % send_interval == 0:
        # Remove timestamps before sending
        data_to_send = data.copy()

        # Serialize and send
        packed_data = msgpack.packb(data_to_send)
        zmq_socket.send(packed_data)

# Clean up
zmq_socket.close()
context.term()
